In [ ]:
import os

import google.generativeai as genai
import pandas as pd

# Configure Gemini with your API key
genai.configure(api_key="your_api_key") # Replace with your actual API key

generation_config = {
    "temperature": 1,
    "top_p": 0.95,
    "top_k": 64,
    "max_output_tokens": 8192,   # 8192 can work, but 2048 is often enough for summaries
    "response_mime_type": "text/plain",
}

model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config=generation_config
)

In [ ]:
chat_session = model.start_chat(
    history=[
        {
            "role": "user",
            "parts": ["You are a market research analyst specializing in consumer electronics.\\n"],
        }
    ]
)

In [ ]:
response1 = chat_session.send_message("What are the key factors customers consider when buying wireless speakers?")
print(response1.text)

response2 = chat_session.send_message("How would you analyze customer feedback about sound quality in speaker reviews?")
print(response2.text)

As a market research analyst specializing in consumer electronics, I've identified several key factors that customers weigh when purchasing wireless speakers. These often interact and their relative importance can shift based on the customer's primary use case (e.g., portable outdoor use vs. stationary home audio).

Here are the paramount considerations:

1.  **Sound Quality:**
    *   **Clarity and Fidelity:** How crisp and accurate the audio reproduction is across different frequencies.
    *   **Bass Response:** The depth and impact of the low-end frequencies, often a major selling point for many consumers.
    *   **Volume and Power:** How loud the speaker can get without distortion, especially important for larger spaces or outdoor use.
    *   **Soundstage/Immersiveness:** How wide and enveloping the sound feels. Stereo pairing capabilities can enhance this significantly.

2.  **Connectivity & Compatibility:**
    *   **Bluetooth Version:** Newer versions (e.g., Bluetooth 5.0+) o

In [ ]:
df = pd.read_csv("/content/reviews_with_sentiment.csv")

api_key = "your_api_key"

# Function to analyze reviews using Gemini AI
def analyze_reviews(reviews, prompt_message):
    genai.configure(api_key=api_key)
    # reviews: list of strings (review texts)
    # prompt_message: instruction for Gemini
    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        generation_config={
            "temperature": 1,
            "top_p": 0.95,
            "top_k": 64,
            "max_output_tokens": 8192,
            "response_mime_type": "text/plain",
        },
    )

    # Format reviews with numbering (makes Gemini's output clearer)

    chat = model.start_chat(history=[])

    numbered_reviews = [f"{i+1}. {r}" for i, r in enumerate(reviews)]
    prompt = prompt_message + "\n\n" + "\n\n".join(numbered_reviews)
    response = chat.send_message(prompt)
    return response.text

In [ ]:
sample_reviews = df["content"].dropna().sample(30, random_state=42).tolist()

prompt_message = "Analyze the following wireless speaker reviews and summarize key themes, top positives, and top complaints:"
insights = analyze_reviews(sample_reviews, prompt_message)
print(insights)

Based on the provided reviews for various wireless speakers (including mentions of Sony, JBL, Beats, and general Bluetooth speakers), here's a summary of key themes, top positives, and top complaints:

## Wireless Speaker Review Summary

### Key Themes:

1.  **Sound Performance vs. Size:** A dominant theme is the impressive sound quality and volume output relative to the speaker's compact or portable size. There's a consistent discussion around bass, with many praising it, but some noting limitations.
2.  **Durability & Portability for Active Use:** The speakers are frequently highlighted for their ruggedness, water/dust/shock resistance, and good battery life, making them ideal for outdoor, travel, or demanding environments.
3.  **Connectivity & Smart Features:** Ease of Bluetooth pairing, multi-device connectivity, and the utility of companion apps (for EQ, settings, multi-speaker pairing) are important aspects, though sometimes with compatibility issues.
4.  **Value Proposition:** M

### Advanced Analysis Techniques

In [ ]:
# Simple short name: first 5 words of product_name
df["short_name"] = df["product_name"].astype(str).apply(lambda x: " ".join(x.split()[:5]))
df[["product_name", "short_name", "brand"]].head()

,product_name,short_name,brand
0,Beats Pill - Portable Bluetooth Speaker (2024 ...,Beats Pill - Portable Bluetooth,Beats
1,Beats Pill - Portable Bluetooth Speaker (2024 ...,Beats Pill - Portable Bluetooth,Beats
2,Beats Pill - Portable Bluetooth Speaker (2024 ...,Beats Pill - Portable Bluetooth,Beats
3,Beats Pill - Portable Bluetooth Speaker (2024 ...,Beats Pill - Portable Bluetooth,Beats
4,Beats Pill - Portable Bluetooth Speaker (2024 ...,Beats Pill - Portable Bluetooth,Beats


In [ ]:
from textblob import TextBlob

def add_sentiment_info(df):
    # Add sentiment score (polarity)
    df["sentiment_score"] = df["content"].apply(lambda x: TextBlob(str(x)).sentiment.polarity)

    # Categorize sentiments
    df["sentiment_category"] = df["sentiment_score"].apply(
        lambda x: "Positive" if x > 0.1
        else "Negative" if x < -0.1
        else "Neutral"
    )
    return df

# Add sentiment info
df = add_sentiment_info(df)

# Quick view of sentiment distribution
print("\nSentiment Distribution:")
print(df["sentiment_category"].value_counts()*100/len(df))


Sentiment Distribution:
sentiment_category
Positive    79.979980
Neutral     18.318318
Negative     1.701702
Name: count, dtype: float64


In [ ]:
# Basic product analysis (Beats + competitors)
product = df["product_name"].value_counts().index[0]
product_reviews = df[df["product_name"] == product]["content"].dropna().tolist()[:30]

prompt = f"""Analyze these reviews for: {product}
1) Top 3 most mentioned features
2) What customers love most
3) Most common complaints
4) Any comments about value for money

Give bullet points and include a few short example phrases."""
print(analyze_reviews(product_reviews, prompt))

Here's an analysis of the JBL Xtreme 3 reviews:

### 1) Top 3 most mentioned features

*   **Sound Quality:** Universally discussed, often praised for its power and clarity.
    *   "Sounds great, punchy and crisp"
    *   "Incredible sound coming out of this thing!!!"
    *   "Bass forward while still being clear."
*   **Battery Life / Charging Capabilities:** Frequent mentions of its endurance and flexible charging options.
    *   "Running for 12 hours at a medium volume... battery was still at 20%."
    *   "The speaker has a charge out port, external devices can charge from either the USB A or USB C port."
    *   "Easy setup, great sound and battery life so far has been good."
*   **Portability / Strap / Size:** Its transportability, including the integrated strap, is a key point of discussion.
    *   "It's a little awkward without the carrying strap. Added bonus- the strap has a bottle opener."
    *   "Light weight sound is great"
    *   "The strap is NOT a p.o.s it's solid &

In [ ]:
# Specifically for Beats products:
beats_products = df[df["brand"].str.contains("Beats", case=False, na=False)]["product_name"].unique()
print("Beats products:", beats_products)

beats_reviews = df[df["brand"].str.contains("Beats", case=False, na=False)]["content"].dropna().tolist()[:30]

prompt = """Analyze these reviews for Beats wireless speakers:
1) Top praised features
2) Top complaints
3) How customers describe sound quality, bass, and loudness
4) Any comparisons to other brands

Be concise and structured."""
print(analyze_reviews(beats_reviews, prompt))

Beats products: ['Beats Pill - Portable Bluetooth Speaker (2024 release) - Matte Black'
 'Beats Pill - Portable Bluetooth Speaker (2024 release) - Statement Red']
Here's an analysis of the Beats wireless speaker reviews:

---

### Beats Wireless Speakers: Customer Review Analysis

**1. Top Praised Features:**
*   **Apple Ecosystem Integration:** Seamless pairing with iPhone, device switching (like AirPods), Find My support, "quintessentially Apple" design/unboxing.
*   **Sound Quality (Overall):** Crystal clear, balanced, detailed, high-fi, impressive for its size.
*   **Design & Build Quality:** Premium feel, sturdy (survived a drop), compact, portable, stylish, "everything-proof" (IP67 dust/water resistance).
*   **Versatility:** Doubles as a speakerphone (clear calls), USB-C for lossless audio playback (from laptop) and device charging.
*   **Battery Life:** Long-lasting (claimed 24 hours).
*   **Value:** Considered "amazing for the price point" and "worth every penny."
*   **Stereo

In [ ]:
if "Sentiment" in df.columns:
    beats_negative = df[
        (df["brand"].str.contains("Beats", case=False, na=False)) &
        (df["Sentiment"] == "Negative")
    ]["content"].dropna().tolist()[:30]

    prompt = """Analyze these NEGATIVE Beats wireless speaker reviews:
1) What specific aspects cause negative reactions?
2) Common failure points (battery, Bluetooth, build quality, sound distortion, etc.)
3) Any repeated phrases that signal frustration
4) What improvements customers suggest

Use bullets and include a few short example phrases."""
    print(analyze_reviews(beats_negative, prompt))

Based on the extremely limited data provided (two identical reviews), the analysis below focuses solely on the single reported issue:

---

### Analysis of NEGATIVE Beats Wireless Speaker Reviews

**1) What specific aspects cause negative reactions?**
*   **Intermittent Operation:** The primary cause of negative reaction is the speaker's inability to maintain consistent power. It unexpectedly shuts down during use.
    *   *Example Phrase:* "Randomly turns on and off"
*   **Limited Usability Window:** The problem consistently occurs after a specific duration of use (30-40 minutes), making the speaker unreliable for longer listening sessions.
    *   *Example Phrase:* "after 30 to 40 minutes of use"
*   **Unpredictability:** The "random" nature of the power cycles adds to frustration, as users cannot anticipate when the speaker will fail.

**2) Common failure points (battery, Bluetooth, build quality, sound distortion, etc.)**
*   **Power Stability/Internal Component Failure:** The core

In [ ]:
def analyze_comparison(df, brands, n_per_brand=20):
    reviews_dict = {}
    for b in brands:
        reviews = df[df["brand"].str.contains(b, case=False, na=False)]["content"].dropna().tolist()[:n_per_brand]
        reviews_dict[b] = reviews

    combined = []
    for brand, reviews in reviews_dict.items():
        combined.extend([f"[{brand}] {r}" for r in reviews])

    prompt = f"""Compare customer feedback across these brands of wireless speakers: {", ".join(brands)}

For EACH brand, summarize:
- Standout strengths
- Common complaints
- Sound quality descriptions (bass/clarity/loudness)
- Battery & connectivity feedback
- Value-for-money perception

Then conclude:
- Where Beats is clearly stronger
- Where Beats is weaker vs competitors
- 3 actionable improvements Beats should prioritize

Use bullets and keep it easy to copy into a report."""
    return analyze_reviews(combined, prompt)

print(analyze_comparison(df, ["Beats", "JBL", "Bose", "Sony"], n_per_brand=15))

Here's a comparison of customer feedback across Beats, JBL, Bose, and Sony wireless speakers:

---

### **Beats (Primarily Beats Pill 2024)**

*   **Standout strengths**
    *   Exceptional Apple ecosystem integration (instant pairing animations, Find My functionality, effortless device switching akin to AirPods).
    *   Premium, sturdy, and stylish design, with some users noting it survived drops.
    *   Multi-functional: acts as a high-quality speakerphone for calls and can play lossless audio via USB-C connectivity.
    *   IP67 rating provides excellent dust and water resistance, suitable for various environments.
    *   Ability to pair two units for stereo sound (Amplify/Stereo modes).
*   **Common complaints**
    *   Lack of a dedicated companion app for advanced controls (e.g., EQ adjustments, detailed battery life monitoring, settings management).
    *   Bass quality can degrade, becoming muddy or experiencing a "flutter" sound at volumes above 50%.
    *   Sound might be 

In [ ]:
def analyze_feature_mentions(df, feature_regex, feature_name, n=40):
    subset = df[df["content"].str.contains(feature_regex, case=False, na=False)]["content"].dropna().tolist()[:n]

    prompt = f"""Analyze mentions of {feature_name} in these wireless speaker reviews:
1) How customers describe {feature_name}
2) What is praised vs criticized
3) Any repeated problems or patterns
4) Suggestions customers imply or explicitly mention

Summarize as actionable bullets."""
    return analyze_reviews(subset, prompt)

print(analyze_feature_mentions(df, r"battery|charge|charging|power", "battery life", n=35))
print(analyze_feature_mentions(df, r"bluetooth|connection|pair|disconnect|latency", "Bluetooth connectivity", n=35))
print(analyze_feature_mentions(df, r"bass|low end|sub|thump", "bass performance", n=35))

Here's an analysis of battery life mentions in the wireless speaker reviews, summarized as actionable bullets:

**1. How Customers Describe Battery Life:**

*   **Generally Positive & Long-Lasting:** Many customers describe it as "long," "great," "good," "solid," or "lasts for hours" even with heavy use.
*   **High Expectations Set by Marketing:** The advertised "24 hrs" is a significant talking point, often described as "a whopping," "mind blowing" (if true), or "about a day's worth."
*   **Skepticism & Real-World Discrepancy:** Several users express skepticism ("testing that claim," "take it with a grain of salt") or explicitly note a shorter actual duration (e.g., 6.5 hours at 60-70% volume with bass-heavy music) compared to the 24-hour claim.
*   **Fast Charging & Modern Port:** Praised for "charges very quickly" and using an "updated USB-C charging port."
*   **Power Bank Functionality:** The ability to "charge your phone" is noted as a "cool" and "value-added" feature.
*   **Util

In [ ]:
# Saving the Gemini outputs
insight_text = analyze_comparison(df, ["Beats", "JBL", "Bose", "Sony"], n_per_brand=15)

with open("gemini_insights_comparison.txt", "w") as f:
    f.write(insight_text)

print("Saved: gemini_insights_comparison.txt")

Saved: gemini_insights_comparison.txt
